# Operating on tensors

_PyTorch (a complete course)_

**Add, multiply, broadcast, matmul, reduce, and reshape — the moves every model is built from.**

A tensor is just an n-dimensional grid of numbers with a shape (its size along each axis, e.g. (batch, features)).

       Almost everything you do is one of five verbs: apply a function to each number, line two tensors up and combine them, multiply matrices, collapse an axis, or rearrange the grid.

---

This notebook is a practice scaffold for the **Operating on tensors** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch

torch.manual_seed(0)

# ---- 1. BROADCASTING: (3,1) + (1,4) -> (3,4) ----
col = torch.tensor([[10.], [20.], [30.]])   # shape (3,1) -- a column
row = torch.tensor([[1., 2., 3., 4.]])      # shape (1,4) -- a row
grid = col + row                            # broadcasts: col repeats across cols, row down rows
print("col", tuple(col.shape), " row", tuple(row.shape), " grid", tuple(grid.shape))
print(grid)                                 # top-left 11, bottom-right 34

# ---- 2. ELEMENTWISE * vs MATRIX MULTIPLY @ ----
A = torch.arange(6.).reshape(2, 3)          # (2,3)
B = torch.ones(2, 3)                        # (2,3)
print("A * B (elementwise) ->", tuple((A * B).shape))   # (2,3): number-by-number
C = torch.ones(3, 2)                        # (3,2)
print("A @ C (matmul)      ->", tuple((A @ C).shape))   # (2,2): inner 3 cancels
# torch.matmul is the same as @; bmm does it per batch element:
batch = torch.randn(8, 2, 3)               # 8 matrices of (2,3)
w     = torch.randn(8, 3, 5)               # 8 matrices of (3,5)
print("bmm batch           ->", tuple(torch.bmm(batch, w).shape))  # (8,2,5)

# ---- 3. REDUCTION over a chosen dim, with / without keepdim ----
row_sum      = grid.sum(dim=1)                  # collapse columns -> (3,)
row_sum_keep = grid.sum(dim=1, keepdim=True)    # keep axis as 1 -> (3,1)
print("sum dim=1          ->", tuple(row_sum.shape), row_sum.tolist())
print("sum dim=1 keepdim  ->", tuple(row_sum_keep.shape))
# keepdim lets it broadcast back (this is exactly the softmax-normalize trick):
normalized = grid / row_sum_keep                # (3,4) / (3,1) -> (3,4), rows sum to 1
print("each row sums to 1 ->", normalized.sum(dim=1).tolist())
print("argmax over dim=1  ->", grid.argmax(dim=1).tolist())  # index of max per row

# ---- 4. RESHAPING: view / reshape / permute / unsqueeze ----
x = torch.arange(24).reshape(2, 3, 4)           # (2,3,4)
print("x                  ->", tuple(x.shape))
print("x.view(2, 12)      ->", tuple(x.view(2, 12).shape))      # contiguous: view ok
xp = x.permute(0, 2, 1)                          # (2,4,3): now non-contiguous
print("x.permute(0,2,1)   ->", tuple(xp.shape), "contiguous?", xp.is_contiguous())
try:
    xp.view(-1)                                  # fails: non-contiguous
except RuntimeError as e:
    print("xp.view(-1) FAILED -> use reshape/contiguous")
print("xp.reshape(-1)     ->", tuple(xp.reshape(-1).shape))     # (24,): reshape copies
print("x.flatten(1)       ->", tuple(x.flatten(1).shape))       # (2,12): merge last two axes
print("x.unsqueeze(0)     ->", tuple(x.unsqueeze(0).shape))     # (1,2,3,4): add batch axis
print("cat dim=0          ->", tuple(torch.cat([x, x], dim=0).shape))   # (4,3,4)
print("stack dim=0        ->", tuple(torch.stack([x, x], dim=0).shape)) # (2,2,3,4) new axis

## Visualize the data & results

_When a (3,1) column broadcasts against a (1,4) row, what (3,4) grid comes out — and what do the per-row sums look like?_

In [ ]:
import numpy as np

col = np.array([[10], [20], [30]])   # shape (3,1) -- a column
row = np.array([[1, 2, 3, 4]])       # shape (1,4) -- a row

grid = col + row                     # broadcasts to (3,4); matches torch exactly
print("grid shape", grid.shape)
print(grid)
# [[11 12 13 14]
#  [21 22 23 24]
#  [31 32 33 34]]

row_sum = grid.sum(axis=1)           # reduce over columns -> (3,)
print("row sums", row_sum)           # [ 50  90 130]

# row_sum keeping the axis (the softmax-normalize trick):
row_sum_keep = grid.sum(axis=1, keepdims=True)  # shape (3,1)
print("keepdims shape", row_sum_keep.shape)     # (3, 1)

# --- the two charts ---
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
im = ax[0].imshow(grid, cmap="viridis")
ax[0].set_xticks(range(4)); ax[0].set_xticklabels([1, 2, 3, 4])
ax[0].set_yticks(range(3)); ax[0].set_yticklabels([10, 20, 30])
ax[0].set_title("(3,1)+(1,4) -> (3,4)")
for i in range(3):
    for j in range(4):
        ax[0].text(j, i, grid[i, j], ha="center", va="center", color="w")
ax[1].bar(["row 0", "row 1", "row 2"], row_sum,
          color=["#4ea1ff", "#7ee787", "#c89bff"])
ax[1].set_title("grid.sum(dim=1) = [50, 90, 130]")
plt.show()

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** You have x of shape (32, 784) (a batch of 32 flattened images) and a weight W of shape (784, 10). Write the linear layer output and give its shape.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Use matmul, not elementwise: z = x @ W. — _A linear layer is a matrix multiply; * would try elementwise and fail (shapes differ)._
- Apply the matmul shape rule: (32,784) @ (784,10), inner 784 matches and cancels. — _Inner dims must match; outer dims survive._

**Answer:** z = x @ W has shape (32, 10) — 10 class scores for each of the 32 images.

</details>

**Problem 2.** Those scores z are (32, 10). Turn them into a predicted class per image.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Reduce over the class axis (the last one): pred = z.argmax(dim=-1). — _argmax returns the index of the largest score; the class axis is dim -1, not the batch axis._

**Answer:** pred = z.argmax(dim=-1) has shape (32,) — one integer class label per image. Reducing over dim=0 by mistake would collapse the batch, which is wrong.

</details>

**Problem 3.** You ran y = x.permute(0, 2, 1) and then y.view(-1) raised an error. Why, and what fixes it?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall permute reorders axes by changing strides, leaving the tensor non-contiguous. — _view needs contiguous row-major memory; permuted data is not._
- Use y.reshape(-1) or y.contiguous().view(-1). — _reshape copies when needed; .contiguous() makes a fresh ordered copy first._

**Answer:** view fails on the non-contiguous permuted tensor. Use y.reshape(-1) (or y.contiguous().view(-1)).

</details>